In [ ]:
YOUR_LANGSMITH_API_KEY

In [ ]:
%pip install -U langsmith openai qdrant-client python-dotenv

: 

In [ ]:
LANGSMITH_TRACING=true
LANGSMITH_API_KEY="YOUR_LANGSMITH_API_KEY"
LANGSMITH_PROJECT=euron-rag-day2

EURI_API_KEY="YOUR_EURI_API_KEY"
EURI_BASE_URL="https://api.euron.one/api/v1/euri"
EURI_CHAT_MODEL="gpt-6-luna"

QDRANT_URL="YOUR_QDRANT_URL" 
QDRANT_API_KEY="YOUR_QDRANT_API_KEY"

In [ ]:
import os
import time
from dotenv import load_dotenv
from openai import OpenAI
from qdrant_client import QdrantClient
from langsmith import traceable, Client
load_dotenv(override=True)
llm = OpenAI(
    api_key=EURI_API_KEY,
    base_url=EURI_BASE_URL
)
qdrant = QdrantClient(
    url="QDRANT_URL",
    api_key="QDRANT_API_KEY"
)
ls_client = Client()
COLLECTION = "promp_embening"
EMBED_MODEL = "gemini-embedding-2"
CHAT_MODEL ="EURI_CHAT_MODEL"
print("LangSmith project:", "LANGSMITH_PROJECT")

In [ ]:
question = "What is Euron Code?"
documents = retrieve_documents(
    question,
    top_k=2
)
for doc in documents:
    print("ID:", doc["id"])
    print("Topic:", doc["topic"])
    print("Similarity:", doc["score"])
    print("Text:", doc["text"])
    print("-" * 50)

In [ ]:
eval_data = [
    {
        "question": "What is Euron Code?",
        "expected": "An AI coding agent for small to medium tasks.",
        "keywords": ["coding", "agent"],
        "relevant_topics": ["coding"],
        "should_abstain": False
    },
    {
        "question": "Who can access Euron Code?",
        "expected": "Euron Plus and Super30 users.",
        "keywords": ["plus", "super30"],
        "relevant_topics": ["coding"],
        "should_abstain": False
    },
    {
        "question": "What subjects does Sudhanshu teach?",
        "expected": "Data science, OpenAI, generative AI, machine learning, deep learning and computer vision.",
        "keywords": ["data science", "machine learning"],
        "relevant_topics": ["teaching"],
        "should_abstain": False
    },{
        "question": "What can users create using Euron tools?",
        "expected": "3D content, images, thumbnails, social posts, banners and PPTs.",
        "keywords": ["images", "thumbnails", "ppt"],
        "relevant_topics": ["features"],
        "should_abstain": False
    },
    {
        "question": "What is Euron's annual revenue?",
        "expected": "I don't know based on the provided context.",
        "keywords": [],
        "relevant_topics": [],
        "should_abstain": True
    }
]

In [ ]:
for item in eval_data:
    docs = retrieve_documents(
        item["question"],
        top_k=3
    )

    print("\nQUESTION:", item["question"])

    for doc in docs:
        print(
            "ID:", doc["id"],
            "Topic:", doc["topic"],
            "Score:", round(doc["score"], 4)
        )
        print("Text:", doc["text"][:200])

In [ ]:
# Replace these with actual IDs after inspecting retrieval.
gold_labels = {
    "What is Euron Code?": {"3"},
    "Who can access Euron Code?": {"3"},
    "What can users create using Euron tools?": {"5"}
}
def precision_at_k(retrieved_ids, relevant_ids, k):
    top_k = retrieved_ids[:k]
    return sum(
        doc_id in relevant_ids
        for doc_id in top_k
    ) / k
def recall_at_k(retrieved_ids, relevant_ids, k):
    if not relevant_ids:
        return None
    return len(
        set(retrieved_ids[:k]) & relevant_ids
    ) / len(relevant_ids)
def reciprocal_rank(retrieved_ids, relevant_ids):
    for rank, doc_id in enumerate(retrieved_ids, 1):
        if doc_id in relevant_ids:
            return 1 / rank
    return 0.0
for question, relevant_ids in gold_labels.items():
    docs = retrieve_documents(question, top_k=3)
    retrieved_ids = [d["id"] for d in docs]
    print("\n", question)
    for k in [1, 2, 3]:
        precision = precision_at_k(
            retrieved_ids, relevant_ids, k
        )
        recall = recall_at_k(
            retrieved_ids, relevant_ids, k
        )
        print(
            f"Top-{k}:",
            f"Precision={precision:.2f}",
            f"Recall={recall:.2f}"
        )
    print(
        "Reciprocal Rank:",
        reciprocal_rank(retrieved_ids, relevant_ids)
    )

In [8]:

import matplotlib.pyplot as plt
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_curve, roc_auc_score
from sklearn.datasets import load_iris
X, y = load_iris(return_X_y=True)
# X, y = make_classification(
#     n_samples=1000,
#     n_features=10,
#     random_state=42
# )
# Step 2: Train/test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.3,
    random_state=42,
    stratify=y
)


# Step 3: Train model
model = LogisticRegression(max_iter=1000)
model.fit(X_train, y_train)

# Step 4: Predict probabilities
y_prob = model.predict_proba(X_test)[:, 1]

# Step 5: Calculate ROC
fpr, tpr, thresholds = roc_curve(y_test, y_prob)
auc = roc_auc_score(y_test, y_prob)

# Step 6: Plot
plt.plot(fpr, tpr, label=f"AUC = {auc:.3f}")
plt.plot([0, 1], [0, 1], "--", label="Random")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate (Recall)")
plt.title("ROC Curve")
plt.legend()
plt.grid()
plt.show()

ValueError: multiclass format is not supported